In [1]:
!git clone https://github.com/cspaper/platform-examples.git

Cloning into 'platform-examples'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 43 (delta 13), reused 36 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 5.71 MiB | 9.56 MiB/s, done.
Resolving deltas: 100% (13/13), done.


In [2]:
%pip install -r platform-examples/requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [56]:
import sys
sys.path.append("../../")
from run_iclr_bench import load_ground_truth

In [57]:
!rm platform-examples/papers/*.pdf

In [60]:
import os
pdfs = os.listdir("/home/wg25r/review_agent/iclr2026_unbalanced/pdfs")
mds =  os.listdir("/home/wg25r/review_agent/iclr2026_unbalanced/papers")
mds = [f.replace(".txt", "") for f in mds]
pdfs = [f.replace(".pdf", "") for f in pdfs]

In [61]:
import os
import openreview
import dotenv
dotenv.load_dotenv()

def get_or_client():
    """Get an authenticated OpenReview API client."""
    import openreview
    username = os.environ.get("OPENREVIEW_USERNAME")
    password = os.environ.get("OPENREVIEW_PASSWORD")
    if not username or not password:
        raise ValueError(
            "Set OPENREVIEW_USERNAME and OPENREVIEW_PASSWORD in .env\n"
            "Sign up at https://openreview.net/signup"
        )
    return openreview.api.OpenReviewClient(
        username=username, password=password,
        baseurl="https://api2.openreview.net",
    )


def download_pdf(or_client, paper_id: str, pdfs_dir: Path = None) -> Path | None:
    """Download a PDF from OpenReview using the authenticated API."""
    outfile = (pdfs_dir or DEFAULT_DATA_DIR / "pdfs") / f"{paper_id}.pdf"
    if outfile.exists() and outfile.stat().st_size > 0:
        return outfile

    try:
        pdf_bytes = or_client.get_pdf(paper_id)
        if len(pdf_bytes) > 1000:
            outfile.write_bytes(pdf_bytes)
            return outfile
        else:
            print(f"    Download failed: got {len(pdf_bytes)} bytes")
            return None
    except Exception as e:
        print(f"    Download error: {e}")
        return None
or_client = get_or_client()

In [65]:
for i in mds:
    if i not in pdfs:
        print(f"Downloading PDF for {i}...")
        download_pdf(or_client, i, Path("/home/wg25r/review_agent/iclr2026_unbalanced/pdfs"))

In [66]:
import os
pdfs = os.listdir("/home/wg25r/review_agent/iclr2026_unbalanced/pdfs")
mds =  os.listdir("/home/wg25r/review_agent/iclr2026_unbalanced/papers")
mds = [f.replace(".txt", "") for f in mds]
pdfs = [f.replace(".pdf", "") for f in pdfs]
assert set(pdfs) == set(mds), "Mismatch between PDFs and parsed papers!"

AssertionError: Mismatch between PDFs and parsed papers!

In [79]:
import json
from pathlib import Path


bench_dir = Path("../../iclr2026_unbalanced")

# Load calibration if provided
calibration_context = ""
cal_dir = ""
calibration_ids = set()

cal_dir_candidate = Path("../../cal")
cal_dir = str(cal_dir_candidate)

# Load excluded IDs
ids_path = Path("../../calibration_ids.json")
if ids_path.exists():
    calibration_ids = set(json.load(open(ids_path)))
    print(f"Excluding {len(calibration_ids)} calibration papers from sampling")

gt_data, papers_dir = load_ground_truth(bench_dir)
print(f"\nLoaded {len(gt_data)} papers from ground truth.")

available = [r for r in gt_data if (papers_dir / f"{r['paper_id']}.txt").exists()]
if calibration_ids:
    available = [r for r in available if r["paper_id"] not in calibration_ids]
print(f"Papers with parsed text (after exclusions): {len(available)}")


import random
random.seed(1343)
samples = random.sample(available, min(100, len(available)))
print(f"Selected {len(samples)} papers (seed={3}).\n")


Excluding 45 calibration papers from sampling

Loaded 100 papers from ground truth.
Papers with parsed text (after exclusions): 98
Selected 98 papers (seed=3).



In [80]:
ids = [sample['paper_id'] for sample in samples]   

In [81]:
print(ids)

['rI2Fa13fUL', 'U6ROetm5nW', 'PFhrOUJZ5o', 'Pa6ak2B9jJ', 'zKQSyT7a7n', 'X2yzXtH4wp', 'MwuSvrthXq', 'iDki7djO2K', 'wUzBBsrdB1', '2EQPpEZtEK', 'NfO2Lt2WY7', 'NFB4QGGS65', 'WwDNiisZQm', 'eETr3lrOQB', 'OuMNJoKJBQ', 'RpDJz00zNh', 'XX5EZoe4ec', 'dCtkwjkK0E', 'crKJJ4Ej60', 'b6qQmQ2F13', 'ey7CXUBn1g', '31CznLfRIS', 'Rt9SeEAMWv', 'ppXAVexrAM', 'Me0n0iESJY', 'wSbVv6xaRr', 'd2pUyiXwcm', 'ZNAY3ivd62', 'GRufFX1gAy', 'KsWRLyIAKP', 'iaoAKDRAJQ', 'oh9ChF7Pv0', 'c2ozZYoZFd', 'j3htU5i01r', 'WhO6Km5Rku', 'tswBfpkwHn', 'bH5M0ts8Y6', '1E4Bltg6Xb', 'khHNHzRjMy', 'ZMzha5gbnF', 'xFo13SaHQm', 'cZFgsLq8Gs', 'bm3rbtEMFj', 'JEN4nsDgh9', 'GMP1S4R6Ke', 'aiM6bRd6bG', 'vGkXf8nvt9', 'oiz0QHejVj', 'c7OsKOOZo8', 'Iq1fNZus2W', 'hQZQVLJrH9', 'IdJakw2jta', 'rBj2iVyrhh', 'Vit5M0G5Gb', 'Kw2mvnzCoc', 'opU91paIvZ', 'pNpnqsn0Si', 'Mz98kwANpF', 'USyGD0eUod', 'piylyBPSau', 'khBHJz2wcV', 'GiaF5cFIpI', '32mrjmaeMP', 'M14YpuTejd', 'XIAta0WOJ6', 'sh1hWO9RHo', 'x6bG2Hoqdf', 'CQ0U1wZYoy', 'mDuTDAK6KU', '1j0ormf8uI', 'ZBhZT307xx', 'D5PJ

In [82]:
for sample in samples:
    os.system(f"cp {Path('../../iclr2026_unbalanced/pdfs') / (sample['paper_id'] + '.pdf')} platform-examples/papers/{sample['paper_id']}.pdf")

In [83]:
!ls platform-examples/papers/ | wc

     98      98    1470


In [ ]:
!cd platform-examples && python agentic-review/submit-batch.py --agent-id="ICLR_main_2026_1" 

In [ ]:
!cd platform-examples && python agentic-review/poll-result-batch.py


Polling 200 job(s) — interval 30s, timeout 1800s

  [COMPLETED] 05hNleYOcG.pdf -> output/05hNleYOcG__ICLR_main_2026_1.md
  [COMPLETED] 0JLUFJMo5p.pdf -> output/0JLUFJMo5p__ICLR_main_2026_1.md
  [COMPLETED] 0K4jNe8ik9.pdf -> output/0K4jNe8ik9__ICLR_main_2026_1.md
  [COMPLETED] 0TmVqOpBbK.pdf -> output/0TmVqOpBbK__ICLR_main_2026_1.md
  [COMPLETED] 0aBAAS0rRT.pdf -> output/0aBAAS0rRT__ICLR_main_2026_1.md
  [COMPLETED] 0lW2UBiEWN.pdf -> output/0lW2UBiEWN__ICLR_main_2026_1.md
  [COMPLETED] 0nvQ5kHXf4.pdf -> output/0nvQ5kHXf4__ICLR_main_2026_1.md
  [COMPLETED] 0xHWd4CUaX.pdf -> output/0xHWd4CUaX__ICLR_main_2026_1.md
  [COMPLETED] 173Pq3F31r.pdf -> output/173Pq3F31r__ICLR_main_2026_1.md
  [COMPLETED] 1PIfB5w05x.pdf -> output/1PIfB5w05x__ICLR_main_2026_1.md
  [COMPLETED] 2eAGrunxVz.pdf -> output/2eAGrunxVz__ICLR_main_2026_1.md
  [COMPLETED] 4VW9HVCRw0.pdf -> output/4VW9HVCRw0__ICLR_main_2026_1.md
  [COMPLETED] 4nOZBufbLC.pdf -> output/4nOZBufbLC__ICLR_main_2026_1.md
  [COMPLETED] 5o0zF03RP9.p

In [22]:
pdf_path_1 = "/home/wg25r/review_agent/iclr2026_unbalanced/pdfs"
pdf_path_2 = "/home/wg25r/review_agent/iclr2026_balanced/pdfs"
output_path = "/home/wg25r/review_agent/new/baselines/cspaper/platform-examples/output"

In [24]:
import os
files_to_zip = [] 
for filename in os.listdir(output_path):
    if filename.endswith(".md"):
        path = os.path.join(pdf_path_1, filename.split("__")[0]+".pdf")
        if not os.path.exists(path):
            path = os.path.join(pdf_path_2, filename.split("__")[0]+".pdf")
            if not os.path.exists(path):
                print(f"Warning: PDF for {filename} not found in either path!")
                continue  
        files_to_zip.append(path)

In [25]:
len(files_to_zip)

200

In [26]:
# zip the files
import zipfile
with zipfile.ZipFile("papers.zip", "w") as zipf:
    for file in files_to_zip:
        zipf.write(file, os.path.basename(file))